Coisine Similarity

In [ ]:
import pandas as pd
import numpy as np


sim_df = pd.read_csv("DE_UKR_aligned_cognate_similarity.csv")

df = sim_df.dropna(subset=["cosine_similarity"]).copy()


df = df.sort_values("cosine_similarity", ascending=False).reset_index(drop=True)


high_thr = df["cosine_similarity"].quantile(0.75)
low_thr  = df["cosine_similarity"].quantile(0.25)

def classify_cognate(sim):
    if sim >= high_thr:
        return "True cognate"
    elif sim <= low_thr:
        return "False cognate"
    else:
        return "Neutral cognate"

df["cognate_type"] = df["cosine_similarity"].apply(classify_cognate)


true_cognates = df[df["cognate_type"] == "True cognate"].copy()
false_cognates = df[df["cognate_type"] == "False cognate"].copy()
neutral_cognates = df[df["cognate_type"] == "Neutral cognate"].copy()


median_sim = df["cosine_similarity"].median()
neutral_cognates["distance_to_median"] = (neutral_cognates["cosine_similarity"] - median_sim).abs()

true_cognates = true_cognates.sort_values("cosine_similarity", ascending=False)
false_cognates = false_cognates.sort_values("cosine_similarity", ascending=True)
neutral_cognates = neutral_cognates.sort_values("distance_to_median", ascending=True)

full_path  = "Cognate_Types_by_Semantic_Similarity.csv"
excel_path = "Cognate_Types_by_Semantic_Similarity.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="All", index=False)
    true_cognates.to_excel(writer, sheet_name="True_cognates", index=False)
    false_cognates.to_excel(writer, sheet_name="False_cognates", index=False)
    neutral_cognates.to_excel(writer, sheet_name="Neutral_cognates", index=False)


print("Thresholds:")
print("Low threshold :", round(low_thr, 4))
print("High threshold:", round(high_thr, 4))
print()

print("Top 10 TRUE cognates")
print(true_cognates[["DE", "UKR", "cosine_similarity"]].head(10))
print()

print("Top 10 FALSE cognates")
print(false_cognates[["DE", "UKR", "cosine_similarity"]].head(10))
print()

print("Top 10 NEUTRAL cognates")
print(neutral_cognates[["DE", "UKR", "cosine_similarity"]].head(10))
print()

print("Saved:")
print(full_path)
print(excel_path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Если sim_df уже есть в памяти, используйте его.
# Иначе:
# sim_df = pd.read_csv("/Users/inna/Desktop/Bailuk_Inna/DE_UKR_aligned_cognate_similarity.csv")

df = sim_df.dropna(subset=["cosine_similarity"]).copy()

# 1. Классификация по квартилям
q25 = df["cosine_similarity"].quantile(0.25)
q75 = df["cosine_similarity"].quantile(0.75)
median_sim = df["cosine_similarity"].median()

def classify_cognate(sim):
    if sim >= q75:
        return "True cognate"
    elif sim <= q25:
        return "False cognate"
    else:
        return "Neutral cognate"

df["cognate_type"] = df["cosine_similarity"].apply(classify_cognate)


df["pair"] = df["DE"] + " — " + df["UKR"]


true_top = (
    df[df["cognate_type"] == "True cognate"]
    .sort_values("cosine_similarity", ascending=False)
    .head(15)
)

false_top = (
    df[df["cognate_type"] == "False cognate"]
    .sort_values("cosine_similarity", ascending=True)
    .head(15)
)

neutral_df = df[df["cognate_type"] == "Neutral cognate"].copy()
neutral_df["distance_to_median"] = (neutral_df["cosine_similarity"] - median_sim).abs()

neutral_top = (
    neutral_df
    .sort_values("distance_to_median", ascending=True)
    .head(15)
    .sort_values("cosine_similarity", ascending=False)
)

sns.set_theme(style="whitegrid", font_scale=1.0)
fig, axes = plt.subplots(1, 3, figsize=(22, 9), sharex=False)

sns.barplot(
    data=true_top,
    y="pair",
    x="cosine_similarity",
    color="#4daf4a",
    ax=axes[0]
)
axes[0].set_title("Top-15 True Cognates")
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("")


sns.barplot(
    data=false_top,
    y="pair",
    x="cosine_similarity",
    color="#e41a1c",
    ax=axes[1]
)
axes[1].set_title("Top-15 False Cognates")
axes[1].set_xlabel("Cosine similarity")
axes[1].set_ylabel("")


sns.barplot(
    data=neutral_top,
    y="pair",
    x="cosine_similarity",
    color="#377eb8",
    ax=axes[2]
)
axes[2].set_title("Top-15 Neutral Cognates")
axes[2].set_xlabel("Cosine similarity")
axes[2].set_ylabel("")


for ax in axes:
    ax.axvline(median_sim, color="gray", linestyle="--", linewidth=1)
    ax.grid(axis="x", linestyle=":", alpha=0.6)

plt.suptitle("Semantic Classification of DE–UKR Cognate Pairs", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
Frequency_Semantic similarity

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr, kruskal


freq_path = "class1_output..csv"
sim_path = "DE_UKR_aligned_cognate_similarity.csv"


freq_df = pd.read_csv(freq_path)
sim_df = pd.read_csv(sim_path)


df = freq_df.merge(
    sim_df[["DE", "UKR", "cosine_similarity"]],
    on=["DE", "UKR"],
    how="inner"
)

df = df.dropna(subset=["freq_DE", "freq_UKR"]).copy()

df["log_freq_DE"] = np.log10(df["freq_DE"] + 1)
df["log_freq_UKR"] = np.log10(df["freq_UKR"] + 1)
df["mean_log_freq"] = (df["log_freq_DE"] + df["log_freq_UKR"]) / 2

if "log_diff" not in df.columns or df["log_diff"].isna().all():
    df["log_diff"] = np.log2((df["freq_DE"] + 1) / (df["freq_UKR"] + 1))

df["abs_log_diff"] = df["log_diff"].abs()

threshold = 0.5
df["freq_cluster"] = np.select(
    [
        df["log_diff"] < -threshold,
        df["log_diff"] > threshold
    ],
    [
        "UKR-dominant",
        "DE-dominant"
    ],
    default="Balanced"
)


df = df.replace([np.inf, -np.inf], np.nan)


variables = ["log_freq_DE", "log_freq_UKR", "mean_log_freq", "log_diff", "abs_log_diff"]
results = []

for var in variables:
    sub = df[[var, "cosine_similarity"]].dropna().copy()

    if len(sub) < 3 or sub[var].nunique() < 2 or sub["cosine_similarity"].nunique() < 2:
        results.append({
            "variable": var,
            "n": len(sub),
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "spearman_rho": np.nan,
            "spearman_p": np.nan
        })
        continue

    pearson_r, pearson_p = pearsonr(sub[var], sub["cosine_similarity"])
    spearman_rho, spearman_p = spearmanr(sub[var], sub["cosine_similarity"])

    results.append({
        "variable": var,
        "n": len(sub),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p
    })

corr_df = pd.DataFrame(results).sort_values("spearman_rho", ascending=False)

print("=== Correlation results ===")
display(corr_df.round(4))


cluster_summary = (
    df.groupby("freq_cluster")["cosine_similarity"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

print("\n=== Semantic similarity by frequency cluster ===")
display(cluster_summary.round(4))


balanced = df.loc[df["freq_cluster"] == "Balanced", "cosine_similarity"].dropna()
ukr_dom = df.loc[df["freq_cluster"] == "UKR-dominant", "cosine_similarity"].dropna()
de_dom = df.loc[df["freq_cluster"] == "DE-dominant", "cosine_similarity"].dropna()

if len(balanced) > 1 and len(ukr_dom) > 1 and len(de_dom) > 1:
    H, p_kw = kruskal(balanced, ukr_dom, de_dom)
    print(f"\nKruskal-Wallis H = {H:.4f}, p = {p_kw:.4f}")
else:
    H, p_kw = np.nan, np.nan
    print("\nKruskal-Wallis test could not be computed (too few observations).")


sns.set_theme(style="whitegrid", font_scale=1.05)
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

plot_vars = [
    ("log_freq_DE", "German frequency"),
    ("log_freq_UKR", "Ukrainian frequency"),
    ("mean_log_freq", "Mean pair frequency"),
    ("abs_log_diff", "Frequency asymmetry")
]

for ax, (var, title) in zip(axes.flatten(), plot_vars):
    sub = df[[var, "cosine_similarity"]].dropna().copy()

    sns.regplot(
        data=sub,
        x=var,
        y="cosine_similarity",
        scatter_kws={"alpha": 0.65, "s": 45},
        line_kws={"color": "darkred"},
        ax=ax
    )
    ax.set_title(f"{title} vs semantic similarity")
    ax.set_ylabel("Cosine similarity")

    if var == "log_freq_DE":
        ax.set_xlabel("log10(freq_DE + 1)")
    elif var == "log_freq_UKR":
        ax.set_xlabel("log10(freq_UKR + 1)")
    elif var == "mean_log_freq":
        ax.set_xlabel("Mean log frequency")
    elif var == "abs_log_diff":
        ax.set_xlabel("|log2(freq_DE / freq_UKR)|")

plt.tight_layout()
plt.show()


order = ["Balanced", "UKR-dominant", "DE-dominant"]

plt.figure(figsize=(9, 6))
sns.boxplot(
    data=df,
    x="freq_cluster",
    y="cosine_similarity",
    order=order,
    palette=["#7aa6c2", "#f28e8e", "#8ec07c"]
)
sns.swarmplot(
    data=df,
    x="freq_cluster",
    y="cosine_similarity",
    order=order,
    color="black",
    alpha=0.65,
    size=3
)

plt.title("Semantic similarity across frequency symmetry clusters")
plt.xlabel("Frequency cluster")
plt.ylabel("Cosine similarity")
plt.tight_layout()
plt.show()



for _, row in corr_df.iterrows():
    var = row["variable"]
    rho = row["spearman_rho"]
    p = row["spearman_p"]

    if pd.isna(rho):
        print(f"{var}: insufficient data")
        continue

    if p < 0.05:
        if rho > 0:
            direction = "positive"
        elif rho < 0:
            direction = "negative"
        else:
            direction = "no"

        print(f"{var}: {direction} association with semantic similarity (Spearman rho={rho:.3f}, p={p:.4f})")
    else:
        print(f"{var}: no statistically significant association (Spearman rho={rho:.3f}, p={p:.4f})")

if not pd.isna(p_kw):
    if p_kw < 0.05:
        print(f"Frequency clusters differ significantly in semantic similarity (Kruskal-Wallis p={p_kw:.4f})")
    else:
        print(f"Frequency clusters do not differ significantly in semantic similarity (Kruskal-Wallis p={p_kw:.4f})")

In [125]:
from Levenshtein import distance as lev_distance

In [127]:
from Levenshtein import distance as lev_distance

cognate_df['edit_distance'] = cognate_df.apply(
    lambda x: lev_distance(x['word1'], x['word2']),
    axis=1
)

df_f = cognate_df.copy()

# нормируем UMAP distance
df_f['umap_norm'] = (
    df_f['distance_umap'] - df_f['distance_umap'].min()
) / (
    df_f['distance_umap'].max() - df_f['distance_umap'].min()
)

# нормируем edit distance
df_f['edit_norm'] = (
    df_f['edit_distance'] - df_f['edit_distance'].min()
) / (
    df_f['edit_distance'].max() - df_f['edit_distance'].min()
)

In [ ]:
!pip install plotly

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("DE_UKR_aligned_cognate_similarity.csv")
df = df.dropna(subset=["freq_DE", "freq_UKR", "cosine_similarity"]).copy()

if "log_diff" not in df.columns or df["log_diff"].isna().all():
    df["log_diff"] = np.log2((df["freq_DE"] + 1) / (df["freq_UKR"] + 1))

threshold = 0.5
df["freq_cluster"] = np.select(
    [
        df["log_diff"] < -threshold,
        df["log_diff"] > threshold
    ],
    [
        "UKR-dominant",
        "DE-dominant"
    ],
    default="Balanced"
)

order = ["Balanced", "UKR-dominant", "DE-dominant"]
palette = {
    "Balanced": "#7aa6c2",
    "UKR-dominant": "#f28e8e",
    "DE-dominant": "#8ec07c"
}

sns.set_theme(style="whitegrid", font_scale=1.05)
fig, ax = plt.subplots(figsize=(12, 8))

sns.violinplot(
    data=df,
    x="freq_cluster",
    y="log_diff",
    order=order,
    hue="freq_cluster",
    palette=palette,
    inner=None,
    cut=0,
    linewidth=1.2,
    legend=False,
    ax=ax
)

x_map = {cat: i for i, cat in enumerate(order)}
x_jitter = df["freq_cluster"].map(x_map).astype(float) + np.random.uniform(-0.18, 0.18, len(df))

sc = ax.scatter(
    x=x_jitter,
    y=df["log_diff"],
    c=df["cosine_similarity"],
    cmap="viridis",
    s=35,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.3
)

ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.axhline(threshold, color="gray", linestyle=":", linewidth=1)
ax.axhline(-threshold, color="gray", linestyle=":", linewidth=1)

ax.set_xticks(range(len(order)))
ax.set_xticklabels(order)
ax.set_xlabel("Cognate frequency cluster")
ax.set_ylabel("log2(freq_DE / freq_UKR)")
ax.set_title("Frequency Symmetry of Cognates with Semantic Similarity", pad=15)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Cosine similarity")

plt.tight_layout()
plt.show()

core–periphery–flexibility

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import plotly.graph_objects as go



pairs = pd.read_csv("/DE_UKR_aligned_cognate_similarity.csv")
pairs = pairs.dropna(subset=["cosine_similarity"]).copy()



def normalize_de(word):
    w = str(word).lower()
    w = (w.replace("ä", "ae")
          .replace("ö", "oe")
          .replace("ü", "ue")
          .replace("ß", "ss"))
    return w

pairs['de_norm'] = pairs['DE'].apply(normalize_de)
pairs['uk_norm'] = pairs['UKR'].str.lower()

print(f"Pairs loaded: {len(pairs)}")


pairs['similarity'] = pairs['cosine_similarity']

q75 = pairs['similarity'].quantile(0.75)
q25 = pairs['similarity'].quantile(0.25)

def label_zone(sim):
    if sim >= q75:
        return "core"
    elif sim <= q25:
        return "periphery"
    else:
        return "middle"

pairs['zone'] = pairs['similarity'].apply(label_zone)



pairs['semantic_shift'] = 1 - pairs['similarity']



color_map = {"core": "green", "middle": "gray", "periphery": "red"}
fig = go.Figure()

for zone in ["core", "middle", "periphery"]:
    subset = pairs[pairs['zone'] == zone]
    fig.add_trace(go.Scatter(
        x=subset['similarity'],
        y=subset['log_diff'].abs() if 'log_diff' in pairs.columns else subset['semantic_shift'],
        mode='markers',
        name=zone,
        marker=dict(size=10, color=color_map[zone], opacity=0.75),
        customdata=subset[['DE', 'UKR']].values,
        hovertemplate=(
            "<b>DE:</b> %{customdata[0]}<br>"
            "<b>UK:</b> %{customdata[1]}<br>"
            "<b>Similarity:</b> %{x:.3f}<br>"
            "<extra></extra>"
        )
    ))

fig.update_layout(
    title="Core–Periphery Structure (DE–UK Cognates)",
    xaxis_title="Cross-lingual cosine similarity",
    yaxis_title="Frequency asymmetry |log2(DE/UK)|",
    legend_title="Zone",
    template="plotly_white"
)

fig.show()

In [31]:
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
import plotly.graph_objects as go



pairs = pd.read_csv("DE_UKR_aligned_cognate_similarity.csv")
pairs = pairs.dropna(subset=["cosine_similarity"]).copy()

def normalize_de(word):
    w = str(word).lower()
    return (w.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss"))

pairs['de_norm'] = pairs['DE'].apply(normalize_de)
pairs['uk_norm'] = pairs['UKR'].str.lower()



print("Loading models...")
de_model = KeyedVectors.load_word2vec_format(
    "de_aligned_cognates.vec"
)
uk_model = KeyedVectors.load_word2vec_format(
    "uk_aligned_cognates.vec"
)
print(f"  DE: {len(de_model)} words | UK: {len(uk_model)} words")



pairs['similarity'] = pairs['cosine_similarity']

q75 = pairs['similarity'].quantile(0.75)
q25 = pairs['similarity'].quantile(0.25)

def label_zone(sim):
    if sim >= q75:   return "core"
    elif sim <= q25: return "periphery"
    else:            return "middle"

pairs['zone'] = pairs['similarity'].apply(label_zone)



def compute_flexibility(model, word, k=10):
    try:
        neighbors = model.most_similar(word, topn=k)
        return np.std([1 - sim for _, sim in neighbors])
    except:
        return np.nan

pairs['flex_de'] = pairs['de_norm'].apply(
    lambda w: compute_flexibility(de_model, w) if w in de_model else np.nan
)
pairs['flex_uk'] = pairs['uk_norm'].apply(
    lambda w: compute_flexibility(uk_model, w) if w in uk_model else np.nan
)
pairs['flexibility'] = pairs[['flex_de', 'flex_uk']].mean(axis=1)

print(f"Pairs with flexibility: {pairs['flexibility'].notna().sum()} / {len(pairs)}")



color_map = {"core": "#2ca02c", "middle": "#7f7f7f", "periphery": "#d62728"}

fig2 = go.Figure()

for zone in ["core", "middle", "periphery"]:
    subset = pairs[pairs['zone'] == zone]
    fig2.add_trace(go.Scatter(
        x=subset['similarity'],
        y=subset['flexibility'],
        mode='markers',
        name=zone,
        marker=dict(size=10, color=color_map[zone], opacity=0.75),
        customdata=subset[['DE', 'UKR']].values,
        hovertemplate=(
            "<b>DE:</b> %{customdata[0]}<br>"
            "<b>UK:</b> %{customdata[1]}<br>"
            "<b>Similarity:</b> %{x:.3f}<br>"
            "<b>Flexibility:</b> %{y:.4f}<br>"
            "<extra></extra>"
        )
    ))


fig2.add_vline(x=q25, line_dash="dash", line_color="red",
               annotation_text=f"Q25={q25:.3f}", annotation_position="top left")
fig2.add_vline(x=q75, line_dash="dash", line_color="green",
               annotation_text=f"Q75={q75:.3f}", annotation_position="top right")

fig2.update_layout(
    title="Core–Periphery Structure of DE–UK Cognates",
    xaxis_title="Cross-lingual similarity (DE–UK)",
    yaxis_title="Lexical flexibility",
    legend_title="Zone",
    template="plotly_white",
    font=dict(family="Arial", size=13),
    width=850,
    height=550
)

fig2.show()

Loading models...
  DE: 215 words | UK: 216 words
Pairs with flexibility: 217 / 217


In [34]:
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
import plotly.graph_objects as go

# =========================
# LOAD PAIRS
# =========================

pairs = pd.read_csv("/Users/inna/Desktop/Bailuk_Inna/DE_UKR_aligned_cognate_similarity.csv")
pairs = pairs.dropna(subset=["cosine_similarity"]).copy()

def normalize_de(word):
    w = str(word).lower()
    return (w.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss"))

pairs['de_norm'] = pairs['DE'].apply(normalize_de)
pairs['uk_norm'] = pairs['UKR'].str.lower()

# =========================
# LOAD REDUCED MODELS
# =========================

print("Loading models...")
de_model = KeyedVectors.load_word2vec_format(
    "/Users/inna/Desktop/Bailuk_Inna/de_aligned_cognates.vec"
)
uk_model = KeyedVectors.load_word2vec_format(
    "/Users/inna/Desktop/Bailuk_Inna/uk_aligned_cognates.vec"
)
print(f"  DE: {len(de_model)} words | UK: {len(uk_model)} words")

# =========================
# SIMILARITY + ZONES
# =========================

pairs['similarity'] = pairs['cosine_similarity']

q75 = pairs['similarity'].quantile(0.75)
q25 = pairs['similarity'].quantile(0.25)

def label_zone(sim):
    if sim >= q75:   return "core"
    elif sim <= q25: return "periphery"
    else:            return "middle"

pairs['zone'] = pairs['similarity'].apply(label_zone)

# =========================
# FLEXIBILITY
# =========================

def compute_flexibility(model, word, k=10):
    try:
        neighbors = model.most_similar(word, topn=k)
        return np.std([1 - sim for _, sim in neighbors])
    except:
        return np.nan

pairs['flex_de'] = pairs['de_norm'].apply(
    lambda w: compute_flexibility(de_model, w) if w in de_model else np.nan
)
pairs['flex_uk'] = pairs['uk_norm'].apply(
    lambda w: compute_flexibility(uk_model, w) if w in uk_model else np.nan
)
pairs['flexibility'] = pairs[['flex_de', 'flex_uk']].mean(axis=1)

print(f"Pairs with flexibility: {pairs['flexibility'].notna().sum()} / {len(pairs)}")

# =========================
# SCATTER: Similarity vs Flexibility
# =========================

color_map = {"core": "#2ca02c", "middle": "#7f7f7f", "periphery": "#d62728"}

fig2 = go.Figure()

for zone in ["core", "middle", "periphery"]:
    subset = pairs[pairs['zone'] == zone]
    fig2.add_trace(go.Scatter(
        x=subset['similarity'],
        y=subset['flexibility'],
        mode='markers',
        name=zone,
        marker=dict(size=10, color=color_map[zone], opacity=0.75),
        customdata=subset[['DE', 'UKR']].values,
        hovertemplate=(
            "<b>DE:</b> %{customdata[0]}<br>"
            "<b>UK:</b> %{customdata[1]}<br>"
            "<b>Similarity:</b> %{x:.3f}<br>"
            "<b>Flexibility:</b> %{y:.4f}<br>"
            "<extra></extra>"
        )
    ))

# пороговые линии q25 / q75
fig2.add_vline(x=q25, line_dash="dash", line_color="red",
               annotation_text=f"Q25={q25:.3f}", annotation_position="top left")
fig2.add_vline(x=q75, line_dash="dash", line_color="green",
               annotation_text=f"Q75={q75:.3f}", annotation_position="top right")

fig2.update_layout(
    title="Core–Periphery Structure of DE–UK Cognates",
    xaxis_title="Cross-lingual similarity (DE–UK)",
    yaxis_title="Lexical flexibility",
    legend_title="Zone",
    template="plotly_white",
    font=dict(family="Arial", size=13),
    width=850,
    height=550
)

fig2.show()

Loading models...
  DE: 215 words | UK: 216 words
Pairs with flexibility: 217 / 217


In [36]:
fig2.write_html("core_periphery_flexibility_plot.html", 
                include_plotlyjs="cdn",   
                full_html=True)
print("Saved: core_periphery_flexibility_plot.html")

Saved: core_periphery_flexibility_plot.html
